# VulGCL — Phase 2 + Training
**Kaggle GPU Notebook** — Three-branch multimodal C/C++ vulnerability detection

**Steps:**
1. Install dependencies
2. Phase 2: CodeBERT-embed pkl graphs → .pt files (~2 hrs on T4)
3. Train 4 models: Graph-only, Image-only, LLM-only, Full VulGCL

**Before running:** Upload `processed.zip` (the graphs_nx folder) as a Kaggle dataset named `vulgcl-graphs`.

In [ ]:
# ── Cell 1: Install ───────────────────────────────────────────────────────────
!pip install transformers torch-geometric torch-scatter torch-sparse networkx tqdm scikit-learn -q
print("Done")

In [ ]:
# ── Cell 2: Setup ─────────────────────────────────────────────────────────────
import os
import glob
import subprocess
import torch

WORK_DIR = "/kaggle/working"

# Find graphs_nx directly in the dataset (Kaggle auto-extracts zips)
result = subprocess.run(["find", "/kaggle/input", "-type", "d", "-name", "graphs_nx"],
                        capture_output=True, text=True)
PKL_ROOT = result.stdout.strip().split("\n")[0]
if not PKL_ROOT:
    raise FileNotFoundError("graphs_nx not found in /kaggle/input")
print(f"PKL root : {PKL_ROOT}")

# Output .pt files
PT_ROOT = f"{WORK_DIR}/pt_files"
for split in ["train", "validation", "test"]:
    os.makedirs(f"{PT_ROOT}/{split}", exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")
print(f"PT root  : {PT_ROOT}")

In [ ]:
# ── Cell 3: Phase 2 functions (CodeBERT embedding) ───────────────────────────
import pickle
import numpy as np
import networkx as nx
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

IMAGE_SIZE = 100
_CB_TOKENIZER = None
_CB_MODEL = None

def _get_codebert(device):
    global _CB_TOKENIZER, _CB_MODEL
    if _CB_TOKENIZER is None:
        print("Loading CodeBERT...")
        _CB_TOKENIZER = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        _CB_MODEL = AutoModel.from_pretrained("microsoft/codebert-base")
        _CB_MODEL.eval()
    return _CB_TOKENIZER, _CB_MODEL.to(device)

def _embed_codes(codes, device, batch_size=64):
    tok, model = _get_codebert(device)
    vecs = []
    with torch.no_grad():
        for i in range(0, len(codes), batch_size):
            enc = tok(codes[i:i+batch_size], return_tensors="pt",
                      max_length=128, truncation=True, padding="max_length")
            out = model(input_ids=enc["input_ids"].to(device),
                        attention_mask=enc["attention_mask"].to(device))
            vecs.append(out.last_hidden_state[:, 0, :].cpu())
    return torch.cat(vecs, dim=0)  # (N, 768)

def pdg_to_pyg(G, label, device):
    nodes = list(G.nodes())
    if not nodes:
        return Data(x=torch.zeros(1, 768),
                    edge_index=torch.zeros(2, 0, dtype=torch.long),
                    y=torch.tensor([label], dtype=torch.float))
    x = _embed_codes([G.nodes[n].get("code", "") for n in nodes], device)
    n2i = {n: i for i, n in enumerate(nodes)}
    edges = list(G.edges())
    if edges:
        edge_index = torch.tensor([[n2i[s] for s, _ in edges],
                                   [n2i[d] for _, d in edges]], dtype=torch.long)
    else:
        edge_index = torch.zeros(2, 0, dtype=torch.long)
    return Data(x=x, edge_index=edge_index,
                y=torch.tensor([label], dtype=torch.float))

def pdg_to_image(G, x_emb):
    n = G.number_of_nodes()
    if n == 0:
        return torch.zeros(3, IMAGE_SIZE, IMAGE_SIZE)
    nodes = list(G.nodes())
    emb = x_emb[:n].float().numpy()
    deg   = nx.degree_centrality(G)
    close = nx.closeness_centrality(G)
    try:
        katz = nx.katz_centrality(G, alpha=0.01, max_iter=1000)
    except Exception:
        katz = {nd: 0.0 for nd in nodes}
    channels = []
    for c_dict in [deg, close, katz]:
        scores = np.array([c_dict.get(nd, 0.0) for nd in nodes],
                          dtype=np.float32).reshape(-1, 1)
        channels.append(scores * emb)
    mat = np.stack(channels, axis=0)  # (3, N, 768)
    for ch in range(3):
        mx = np.abs(mat[ch]).max()
        if mx > 0:
            mat[ch] /= mx
    t = torch.from_numpy(mat).unsqueeze(0)  # (1, 3, N, 768)
    return F.interpolate(t, size=(IMAGE_SIZE, IMAGE_SIZE),
                         mode="bilinear", align_corners=False).squeeze(0)

def pdg_to_slice(G, top_k=10):
    if G.number_of_nodes() == 0:
        return ""
    cent = nx.betweenness_centrality(G)
    top  = sorted(cent, key=cent.get, reverse=True)[:top_k]
    top.sort(key=lambda nd: G.nodes[nd].get("line", 0))
    return " ".join(G.nodes[nd].get("code", "").strip()
                    for nd in top if G.nodes[nd].get("code", ""))

print("Phase 2 functions ready.")

In [ ]:
# ── Cell 4: Run Phase 2 ───────────────────────────────────────────────────────
def run_phase2(pkl_root, pt_root, splits=("train", "validation", "test")):
    for split in splits:
        pkl_dir = f"{pkl_root}/{split}"
        pt_dir  = f"{pt_root}/{split}"
        os.makedirs(pt_dir, exist_ok=True)
        pkl_files = sorted(glob.glob(f"{pkl_dir}/*.pkl"))
        if not pkl_files:
            print(f"[Phase 2] {split}: no pkl files found at {pkl_dir}")
            continue
        print(f"\n[Phase 2] {split} — {len(pkl_files)} graphs")
        ok = skip = err = 0
        for pkl_path in tqdm(pkl_files, desc=f"  Embed/{split}"):
            stem   = os.path.splitext(os.path.basename(pkl_path))[0]
            pt_out = f"{pt_dir}/{stem}.pt"
            if os.path.exists(pt_out):
                skip += 1
                continue
            try:
                with open(pkl_path, "rb") as f:
                    obj = pickle.load(f)
                G, label  = obj["graph"], obj["label"]
                data      = pdg_to_pyg(G, label, DEVICE)
                img       = pdg_to_image(G, data.x)
                llm_slice = pdg_to_slice(G)
                torch.save({"graph": data, "image": img,
                            "llm_slice": llm_slice, "label": float(label)}, pt_out)
                ok += 1
            except Exception as e:
                err += 1
                tqdm.write(f"  error {stem}: {e}")
        print(f"  ok={ok}  skip={skip}  err={err}")

run_phase2(PKL_ROOT, PT_ROOT)
print("\nPhase 2 complete.")

In [ ]:
# ── Cell 5: Dataset + DataLoaders ────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Batch
from transformers import AutoTokenizer

class VulDataset(Dataset):
    def __init__(self, split, pt_root, max_seq_len=512):
        self.tok         = AutoTokenizer.from_pretrained("microsoft/codebert-base")
        self.max_seq_len = max_seq_len
        self.pt_files    = sorted(glob.glob(f"{pt_root}/{split}/*.pt"))
        print(f"  {split}: {len(self.pt_files)} samples")

    def __len__(self):
        return len(self.pt_files)

    def __getitem__(self, idx):
        obj  = torch.load(self.pt_files[idx], map_location="cpu", weights_only=False)
        text = obj.get("llm_slice", "") or ""
        enc  = self.tok(text, max_length=self.max_seq_len,
                        truncation=True, padding="max_length", return_tensors="pt")
        return {
            "graph":          obj["graph"],
            "image":          obj["image"],
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label":          torch.tensor(obj["label"], dtype=torch.float),
        }

def collate_fn(samples):
    return {
        "graph":          Batch.from_data_list([s["graph"] for s in samples]),
        "image":          torch.stack([s["image"]          for s in samples]),
        "input_ids":      torch.stack([s["input_ids"]      for s in samples]),
        "attention_mask": torch.stack([s["attention_mask"] for s in samples]),
        "label":          torch.stack([s["label"]          for s in samples]),
    }

print("Loading datasets...")
train_ds = VulDataset("train",      PT_ROOT)
val_ds   = VulDataset("validation", PT_ROOT)
test_ds  = VulDataset("test",       PT_ROOT)

BS = 16
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True,  collate_fn=collate_fn, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BS, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BS, shuffle=False, collate_fn=collate_fn, num_workers=2)
print("DataLoaders ready.")

In [ ]:
# ── Cell 6: Models ────────────────────────────────────────────────────────────
import torch.nn as nn
from torch_geometric.nn import GATConv, global_mean_pool
from transformers import RobertaModel

class GraphBranch(nn.Module):
    def __init__(self, in_dim=768, hidden_dim=256, num_layers=2):
        super().__init__()
        self.convs = nn.ModuleList([
            GATConv(in_dim if i == 0 else hidden_dim, hidden_dim)
            for i in range(num_layers)
        ])
        self.relu = nn.ReLU()
    def forward(self, data):
        x, ei, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = self.relu(conv(x, ei))
        return global_mean_pool(x, batch)  # (B, 256)

class ImageBranch(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((4, 4)),
        )
        self.fc = nn.Linear(64 * 4 * 4, hidden_dim)
    def forward(self, x):
        return self.fc(self.cnn(x).view(x.size(0), -1))  # (B, 256)

class LLMBranch(nn.Module):
    def __init__(self, hidden_dim=256):
        super().__init__()
        self.encoder = RobertaModel.from_pretrained("microsoft/codebert-base")
        self.proj    = nn.Linear(768, hidden_dim)
    def forward(self, input_ids, attention_mask):
        cls = self.encoder(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return self.proj(cls)  # (B, 256)

# ── Single-branch classifiers ─────────────────────────────────────────────────
class GraphClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = GraphBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        g = batch["graph"].to(DEVICE)
        return self.head(self.branch(g)).squeeze(-1)

class ImageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = ImageBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        return self.head(self.branch(batch["image"].to(DEVICE))).squeeze(-1)

class LLMClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.branch = LLMBranch()
        self.head   = nn.Sequential(nn.Dropout(0.1), nn.Linear(256, 1))
    def forward(self, batch):
        return self.head(self.branch(
            batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE)
        )).squeeze(-1)

class VulGCLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.graph_branch = GraphBranch()
        self.image_branch = ImageBranch()
        self.llm_branch   = LLMBranch()
        self.head = nn.Sequential(
            nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.1), nn.Linear(256, 1)
        )
    def forward(self, batch):
        h_g = self.graph_branch(batch["graph"].to(DEVICE))
        h_i = self.image_branch(batch["image"].to(DEVICE))
        h_l = self.llm_branch(batch["input_ids"].to(DEVICE),
                              batch["attention_mask"].to(DEVICE))
        return self.head(torch.cat([h_g, h_i, h_l], dim=-1)).squeeze(-1)

print("Models defined.")

In [ ]:
# ── Cell 7: Training utilities ────────────────────────────────────────────────
import json
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

def evaluate(model, loader):
    model.eval()
    preds, labels, probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch).cpu()
            p      = torch.sigmoid(logits)
            preds.extend((p > 0.5).long().tolist())
            labels.extend(batch["label"].long().tolist())
            probs.extend(p.tolist())
    return {
        "f1":  f1_score(labels, preds, zero_division=0),
        "acc": accuracy_score(labels, preds),
        "auc": roc_auc_score(labels, probs) if len(set(labels)) > 1 else 0.0,
    }

def run_experiment(name, model, epochs=10, lr=2e-5):
    print(f"\n{'='*60}")
    print(f"Training : {name}")
    print(f"{'='*60}")
    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss()
    use_amp   = DEVICE == "cuda"
    scaler    = torch.cuda.amp.GradScaler() if use_amp else None
    best_f1, best_state = 0.0, None

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch:02d}", leave=False):
            labels = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            if use_amp:
                with torch.cuda.amp.autocast():
                    loss = criterion(model(batch), labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(batch), labels)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()

        val_m = evaluate(model, val_loader)
        print(f"  Epoch {epoch:02d}  loss={total_loss/len(train_loader):.4f}  "
              f"val_F1={val_m['f1']:.4f}  val_AUC={val_m['auc']:.4f}")
        if val_m["f1"] > best_f1:
            best_f1    = val_m["f1"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    test_m = evaluate(model, test_loader)
    torch.save(best_state, f"{WORK_DIR}/{name}_best.pt")
    print(f"\n  TEST  F1={test_m['f1']:.4f}  AUC={test_m['auc']:.4f}  Acc={test_m['acc']:.4f}")
    return test_m

print("Training utilities ready.")

In [ ]:
# ── Cell 8: Run all experiments ───────────────────────────────────────────────
# Each run saves best model to /kaggle/working/{name}_best.pt
results = {}

# Ablation baselines first (faster, no CodeBERT fine-tuning)
results["graph_only"] = run_experiment("graph_only", GraphClassifier(), epochs=10, lr=1e-4)
results["image_only"] = run_experiment("image_only", ImageClassifier(), epochs=10, lr=1e-4)

# LLM-only on PDG slice (CodeBERT fine-tuned end-to-end)
results["llm_only"]   = run_experiment("llm_only",   LLMClassifier(),   epochs=5,  lr=2e-5)

# Full VulGCL (3-branch fusion)
results["vulgcl"]     = run_experiment("vulgcl",     VulGCLModel(),     epochs=10, lr=2e-5)

In [ ]:
# ── Cell 9: Results summary ───────────────────────────────────────────────────
print("\n" + "="*60)
print("FINAL RESULTS — Devign Test Set")
print("="*60)
print(f"{'Model':<20} {'F1':>8} {'AUC':>8} {'Acc':>8}")
print("-"*46)
for name, m in results.items():
    print(f"{name:<20} {m['f1']:>8.4f} {m['auc']:>8.4f} {m['acc']:>8.4f}")
print("="*60)

with open(f"{WORK_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved: {WORK_DIR}/results.json")